# 03 — Test vol-engine (pub/sub channel `vol_update`)

Smoke test du container `fxvol-vol-engine` — étape 3/5. Valide que **l'engine publie sur le channel Redis `vol_update`** avec le payload identique au cache (`SET latest_vol_surface:EURUSD`), à la cadence du cycle (180s par message).

**Pourquoi c'est critique** : le channel `vol_update` est ce que `api/ws/redis_bridge.py` consume pour pousser la surface au frontend via WebSocket `/ws/vol`. Le notebook 02 a validé le **cache** Redis (`SET latest_vol_surface:EURUSD`), ce notebook valide le **stream** (`PUBLISH vol_update`). Sans ce stream, le smile chart et le tableau des signaux côté frontend ne se rafraîchissent jamais en live.

**Différence majeure avec risk/03** : risk-engine cycle = 2s donc on reçoit 2-3 messages en 5s d'écoute. Vol-engine cycle = **180s** donc une seule fenêtre raisonnable de validation = ~200s pour ≥ 1 message. Pas de check de cadence "intervalle médian" (faudrait écouter > 6 min pour 2 intervalles).

**Stratégie** : 2 modes possibles selon ta patience :
1. **Mode rapide (~5-10s)** : on lit la dernière surface du cache, force un PUBLISH manuel sur le channel pour vérifier que le bridge le voit, et on s'assure qu'il y a un PUBLISH récent corrélé au heartbeat.
2. **Mode complet (~200s)** : subscribe + listen jusqu'à recevoir un vrai PUBLISH du engine.

Ce notebook fait **mode complet** par défaut (waiting passif jusqu'à 200s), avec early-exit dès qu'un message arrive.

**Couvre** :
1. Subscribe `vol_update` + listen jusqu'à 200s → recevoir ≥ 1 message
2. Tous les messages JSON-parseables
3. Schéma identique au cache : `{symbol, timestamp, surface}` au top-level
4. `surface` contient des tenors publics + au moins `_svi`
5. Cohérence cache ↔ stream : le `timestamp` du dernier message reçu == `timestamp` du `latest_vol_surface:EURUSD` lu juste après (preuve d'atomicité SET avant PUBLISH)

**Préreq** :
- Notebook 02 vert (heartbeat frais, surface non-vide en cache)
- Patience : 1-3 minutes pour que le prochain cycle vol publie

## Setup

Pas de seed surface ici (vol-engine n'a pas besoin d'input downstream — il fetch la chain et calcule depuis le spot). On vérifie juste que `latest_spot:EURUSD` est en place pour que le cycle ne skip pas pendant qu'on écoute.

In [1]:
import json
import time
from datetime import datetime, timezone

import redis

REDIS_URL = "redis://localhost:6380/0"
SYMBOL = "EURUSD"
CHANNEL = "vol_update"

# Cycle vol = 180s + ~5s calcul. 200s = 1 cycle nominal complet, marge de
# sécurité raisonnable. Si le notebook 02 vient de tourner OK avec heartbeat
# âgé de quelques secondes, on peut attendre presque 180s avant le prochain.
MAX_LISTEN_S = 200.0
POLL_INTERVAL_S = 0.5
MIN_MESSAGES = 1

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

r = redis.from_url(REDIS_URL, decode_responses=True)
if not r.ping():
    raise RuntimeError("Redis ping FAIL — check 'docker compose ps'")

spot_raw = r.get(f"latest_spot:{SYMBOL}")
if spot_raw is None:
    print("  [WARN] latest_spot:EURUSD absent — seed factice 1.17000")
    r.set(f"latest_spot:{SYMBOL}", "1.17000", ex=600)
else:
    print(f"  [INFO] latest_spot:{SYMBOL} = {spot_raw[:60]}")
print(f"Connected -> {REDIS_URL}, channel = {CHANNEL!r}")

  [INFO] latest_spot:EURUSD = 1.169895
Connected -> redis://localhost:6380/0, channel = 'vol_update'


## 1. Subscribe + listen jusqu'à 200s

**Ce que tu dois voir** : ≥ 1 message reçu en moins de 200s. Avec un heartbeat récent (cf. notebook 02), tu reçois souvent le 1er message en 0-180s.

**Si 0 message en 200s** : engine ne publie plus. Causes possibles :
- `latest_spot:EURUSD` absent → cycle skip (cf. setup ci-dessus)
- IB chain vide (TrustedIPs KO, weekend, perms market data) → `_compute_surface` retourne `{}` → cycle skip
- Crash silencieux du publisher (regarder logs)

Le code fait early-exit dès le 1er message, donc en pratique le test prend bien moins que 200s.

In [2]:
print(f"== 1. subscribe + listen up to {MAX_LISTEN_S}s ==")

sub = redis.from_url(REDIS_URL, decode_responses=True).pubsub(ignore_subscribe_messages=True)
sub.subscribe(CHANNEL)

messages = []  # liste de (timestamp_recv_ms, raw_data)
deadline = time.perf_counter() + MAX_LISTEN_S
start_ts = time.perf_counter()
while time.perf_counter() < deadline and len(messages) < MIN_MESSAGES:
    msg = sub.get_message(timeout=POLL_INTERVAL_S)
    if msg and msg.get("type") == "message":
        messages.append((time.perf_counter() * 1000, msg["data"]))
elapsed = time.perf_counter() - start_ts

sub.unsubscribe(CHANNEL)
sub.close()

record(f"≥ {MIN_MESSAGES} message reçu en {MAX_LISTEN_S}s",
       len(messages) >= MIN_MESSAGES,
       f"{len(messages)} message(s) en {elapsed:.1f}s")

== 1. subscribe + listen up to 200.0s ==
  [OK  ] ≥ 1 message reçu en 200.0s  | 1 message(s) en 120.8s


## 2. JSON valide + schéma `{symbol, timestamp, surface}`

Le payload publié sur `vol_update` est exactement celui écrit dans `latest_vol_surface:EURUSD` (cf. `bus/publisher.py:155` — même `surface_payload` passé à `set` puis `publish`).

Format attendu :
```json
{"symbol":"EURUSD", "timestamp":"ISO", "surface":{"1M":{...}, "_svi":{...}, ...}}
```

In [3]:
print("== 2. JSON parsing + schema ==")

if not messages:
    record("JSON valide + clés top-level", False, "aucun message à inspecter (cf. §1)")
    parsed_msgs = []
else:
    parse_errors = 0
    schema_errors = 0
    parsed_msgs = []
    for _, raw in messages:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            parse_errors += 1
            continue
        if {"symbol", "timestamp", "surface"} <= set(obj.keys()) and isinstance(obj.get("surface"), dict):
            parsed_msgs.append(obj)
        else:
            schema_errors += 1
    record("tous les messages JSON-parseables",
           parse_errors == 0,
           f"{parse_errors} erreurs de parse / {len(messages)}")
    record("top-level keys = {symbol, timestamp, surface}",
           schema_errors == 0,
           f"{schema_errors} schémas incomplets / {len(messages)}")
    if parsed_msgs:
        sample = parsed_msgs[0]
        print(f"  [INFO] sample[0].keys = {sorted(sample.keys())}")
        print(f"  [INFO] sample[0].symbol = {sample.get('symbol')!r}")
        print(f"  [INFO] sample[0].timestamp = {sample.get('timestamp')!r}")
        print(f"  [INFO] sample[0].surface.keys = {sorted(sample.get('surface', {}).keys())}")

== 2. JSON parsing + schema ==
  [OK  ] tous les messages JSON-parseables  | 0 erreurs de parse / 1
  [OK  ] top-level keys = {symbol, timestamp, surface}  | 0 schémas incomplets / 1
  [INFO] sample[0].keys = ['surface', 'symbol', 'timestamp']
  [INFO] sample[0].symbol = 'EURUSD'
  [INFO] sample[0].timestamp = '2026-04-29T09:26:02.335047Z'
  [INFO] sample[0].surface.keys = ['1M', '2M', '3M', '4M', '5M', '6M', '_ssvi', '_svi']


## 3. Surface dans le payload : ≥ 1 tenor public + `_svi`

Vérifie que le payload poussé n'est pas vide / dégradé. Le notebook 02 a déjà validé en détail le contenu de la surface ; ici on s'assure que **le pipeline pub/sub envoie bien la même chose** que ce qu'on lit en cache.

Critère minimal : ≥ 1 tenor public (1M, 2M, ...) + `_svi` non-vide. Les autres champs (`_har`, `_garch`, `_fair_q`, `_ssvi`) sont vérifiés en notebook 02.

In [4]:
print("== 3. surface payload non-dégradée ==")

if not parsed_msgs:
    record("surface payload", False, "skip (cf. §2)")
else:
    obj = parsed_msgs[-1]  # dernier message reçu
    surface = obj.get("surface", {})
    public_tenors = [t for t in surface if not t.startswith("_") and isinstance(surface[t], dict)]
    record("≥ 1 tenor public dans surface du payload",
           len(public_tenors) >= 1,
           f"tenors = {public_tenors}")
    svi = surface.get("_svi") or {}
    record("_svi non-vide dans payload",
           bool(svi),
           f"svi tenors = {sorted(svi)}")
    # Sanity timestamp ISO
    ts = obj.get("timestamp")
    try:
        datetime.fromisoformat(ts.replace("Z", "+00:00"))
        ts_ok = True
    except (ValueError, AttributeError):
        ts_ok = False
    record("timestamp ISO-8601 parsable", ts_ok, f"ts = {ts!r}")

== 3. surface payload non-dégradée ==
  [OK  ] ≥ 1 tenor public dans surface du payload  | tenors = ['1M', '2M', '3M', '4M', '5M', '6M']
  [OK  ] _svi non-vide dans payload  | svi tenors = ['1M', '2M', '3M', '4M', '5M', '6M']
  [OK  ] timestamp ISO-8601 parsable  | ts = '2026-04-29T09:26:02.335047Z'


## 4. Cohérence cache ↔ stream (atomicité SET → PUBLISH)

**Le test clé du pipeline** : `publish_vol_update` fait `SET latest_vol_surface` **AVANT** `PUBLISH vol_update` (cf. `bus/publisher.py:143-155`). Donc juste après réception du message PUBLISH, le `GET latest_vol_surface:EURUSD` doit retourner le **même payload** (timestamp identique).

**Pourquoi ça compte** : si un consumer reçoit un event `vol_update` et fait un `GET latest_vol_surface` immédiatement, il doit trouver les bonnes données. Si `PUBLISH` arrivait avant `SET`, les consumers WS bridge pourraient lire une version périmée — race condition.

**Tolérance** : on compare le `timestamp` string. Si égalité exacte → `SET` puis `PUBLISH` du même payload → atomicité confirmée.

In [5]:
print("== 4. cohérence cache ↔ stream ==")

if not parsed_msgs:
    record("cohérence", False, "skip (cf. §2)")
else:
    last_msg = parsed_msgs[-1]
    msg_ts = last_msg.get("timestamp")
    cache_raw = r.get(f"latest_vol_surface:{SYMBOL}")
    cache_ts = None
    if cache_raw:
        try:
            cache_ts = json.loads(cache_raw).get("timestamp")
        except json.JSONDecodeError:
            pass
    # Le cache peut avoir avancé d'un cycle si on a tardé à inspecter ;
    # accepter l'égalité OU "cache plus récent".
    if cache_ts is None:
        record("cache lisible", False, "latest_vol_surface absent ou JSON invalide")
    elif cache_ts == msg_ts:
        record("timestamp message == timestamp cache (atomicité parfaite)",
               True, f"ts = {msg_ts}")
    else:
        # OK si cache_ts >= msg_ts (le cache a été overwrite par un cycle ultérieur)
        try:
            msg_dt = datetime.fromisoformat(msg_ts.replace("Z", "+00:00"))
            cache_dt = datetime.fromisoformat(cache_ts.replace("Z", "+00:00"))
            ordered_ok = cache_dt >= msg_dt
        except (ValueError, AttributeError):
            ordered_ok = False
        record("cache_ts ≥ msg_ts (un cycle a pu passer entre PUBLISH et notre GET)",
               ordered_ok,
               f"msg={msg_ts}, cache={cache_ts}")

== 4. cohérence cache ↔ stream ==
  [OK  ] timestamp message == timestamp cache (atomicité parfaite)  | ts = 2026-04-29T09:26:02.335047Z


## Récap final

In [6]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<60} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<60} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — pub/sub stream sain. Pass au notebook 04 (WS bridge api).")


LABEL                                                        STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
≥ 1 message reçu en 200.0s                                   OK      1 message(s) en 120.8s
tous les messages JSON-parseables                            OK      0 erreurs de parse / 1
top-level keys = {symbol, timestamp, surface}                OK      0 schémas incomplets / 1
≥ 1 tenor public dans surface du payload                     OK      tenors = ['1M', '2M', '3M', '4M', '5M', '6M']
_svi non-vide dans payload                                   OK      svi tenors = ['1M', '2M', '3M', '4M', '5M', '6M']
timestamp ISO-8601 parsable                                  OK      ts = '2026-04-29T09:26:02.335047Z'
timestamp message == timestamp cache (atomicité parfaite)    OK      ts = 2026-04-29T09:26:02.335047Z
----------------------------------------------------------------------------------------------

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| 0 message en 200s | engine cycle skip permanent | `redis-cli GET latest_spot:EURUSD` (présent ?), `docker logs fxvol-vol-engine --tail 50` (cycle log ?), si `vol_cycle_skipped reason=no_surface` → IB chain vide |
| `parse_errors > 0` | payload non-JSON ou serializer cassé | `redis-cli SUBSCRIBE vol_update` manuel pour voir le payload brut |
| schéma incomplet (pas `{symbol,timestamp,surface}`) | `publish_vol_update` modifié sans aligner les consumers | revue `src/bus/publisher.py:123` |
| `_svi` vide dans payload | aucun tenor n'a fitté SVI ce cycle | normal en early start, sinon investiguer chain IB (notebook 02 §3) |
| `cache_ts < msg_ts` | écriture cache cassée (TTL trop court ?) | check `keys.TTL_VOL_SURFACE` ; sinon bug atomicité publisher |
| reçu 1 message puis silence > 4 min | engine crash ou hang après 1er cycle | `docker logs fxvol-vol-engine --tail 100` ; chercher Traceback ou `IB disconnected` |